## Setup

In [ ]:
import os
os.environ['MPLBACKEND'] = 'agg'
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib-config'

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

## Load Data

In [ ]:
train = pd.read_parquet('dados/feature_matrix_train.parquet')
test = pd.read_parquet('dados/feature_matrix_test.parquet')

print(f'train shape: {train.shape}')
print(f'test shape:  {test.shape}')

## Feature Selection

In [ ]:
NON_FEATURE_COLS = [
    'id', 'date', 'season', 'round', 'home_team', 'away_team',
    'home_score', 'away_score', 'home_state', 'away_state', 'result'
]
FEATURE_COLS = [c for c in train.columns if c not in NON_FEATURE_COLS]

X_train = train[FEATURE_COLS]
y_train = train['result']
X_test = test[FEATURE_COLS]
y_test = test['result']

print(f'Feature columns: {len(FEATURE_COLS)}')

## Model Training

In [ ]:
lr = LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
print('Model trained.')

## Cross-Validation

In [ ]:
tss = TimeSeriesSplit(n_splits=5)
cv_acc = cross_val_score(lr, X_train, y_train, cv=tss, scoring='accuracy')
cv_f1 = cross_val_score(lr, X_train, y_train, cv=tss, scoring='f1_macro')

print(f'CV accuracy: {cv_acc.mean():.3f} +/- {cv_acc.std():.3f}')
print(f'CV macro-F1: {cv_f1.mean():.3f} +/- {cv_f1.std():.3f}')

## Evaluation

In [ ]:
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['HomeWin', 'Draw', 'AwayWin']))

In [ ]:
labels = ['HomeWin', 'Draw', 'AwayWin']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

## Baseline Comparison

In [ ]:
naive_acc = (y_test == 'HomeWin').mean()
model_acc = accuracy_score(y_test, y_pred)
model_f1 = f1_score(y_test, y_pred, average='macro')

print(f'Naive (always HomeWin) accuracy: {naive_acc:.4f}')
print(f'Model test accuracy:             {model_acc:.4f}')
print(f'Model macro-F1:                  {model_f1:.4f}')
print('Note: class_weight=balanced trades raw accuracy for Draw/AwayWin recall. Primary metric is macro-F1.')